# Topic 12: Cross-Validation: Why One Split Is Not Enough, and Why One CV Is Not Either

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 12 lecture note](https://dongchul.kim/csci3320-fall2026/topic/11), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## One split, one number

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target          # 1 = malignant, as in Topic 9b

for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed)
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(max_iter=5000).fit(scaler.transform(X_train), y_train)
    acc = model.score(scaler.transform(X_test), y_test)     # score() is accuracy
    wrong = round((1 - acc) * len(y_test))
    print(f"random_state={seed}:  accuracy {acc:.4f}   ({wrong} of {len(y_test)} wrong)")

## Measure it: 200 splits instead of one

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def holdout_scores(X, y, n_seeds=200):
    """One 80/20 split per seed. Returns the test accuracies and the test-set size."""
    scores = []
    for seed in range(n_seeds):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed)
        scaler = StandardScaler().fit(X_train)
        model = LogisticRegression(max_iter=5000).fit(scaler.transform(X_train), y_train)
        scores.append(model.score(scaler.transform(X_test), y_test))
    return np.array(scores), len(y_test)

cancer, iris = load_breast_cancer(), load_iris()
for name, X, y in [("breast cancer", cancer.data, 1 - cancer.target),
                   ("iris", iris.data, iris.target)]:
    s, m = holdout_scores(X, y)
    print(f"{name}: {len(y)} rows, so every test set has {m}")
    print(f"  worst seed {s.argmin():3d}: {s.min():.4f}     best seed {s.argmax():3d}: {s.max():.4f}")
    print(f"  mean {s.mean():.4f}   standard deviation {s.std():.4f}\n")

### Small data versus big data

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# One population of 100,000 rows: 20 features, two classes, fixed difficulty.
# make_classification shuffles its rows, so the first n rows are a fair sample of it.
X_all, y_all = make_classification(n_samples=100_000, random_state=0)

print("      n   test rows     min     max     std   sqrt(p(1-p)/m)")
for n in [200, 1_000, 10_000, 100_000]:
    X, y = X_all[:n], y_all[:n]
    scores = []
    for seed in range(200):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed)
        model = LogisticRegression().fit(X_train, y_train)
        scores.append(model.score(X_test, y_test))
    s, m = np.array(scores), len(y_test)
    p = s.mean()                                   # the typical accuracy
    print(f"{n:7,d} {m:9,d}    {s.min():.4f}  {s.max():.4f}  {s.std():.4f}      {np.sqrt(p * (1 - p) / m):.4f}")

## 10-fold cross-validation

### By hand first

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target

folds = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
scores, tested = [], []
for k, (train_idx, test_idx) in enumerate(folds.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    scaler = StandardScaler().fit(X_train)            # the 9 training folds only
    model = LogisticRegression(max_iter=5000).fit(scaler.transform(X_train), y_train)
    acc = model.score(scaler.transform(X_test), y_test)
    scores.append(acc)
    tested.extend(test_idx)
    print(f"fold {k:2d}: train on {len(train_idx)}, test on {len(test_idx)} "
          f"({y_test.sum()} malignant)   accuracy {acc:.4f}")

scores = np.array(scores)
print("\nevery patient tested exactly once:", sorted(tested) == list(range(len(y))))
print(f"10-fold CV accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

### The same thing in two lines

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
folds = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=folds)

print(scores.round(4))
print(f"10-fold CV accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

### Why stratified folds

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import KFold, StratifiedKFold

data = load_breast_cancer()
X, y = data.data, 1 - data.target
print(f"the whole data set: {y.sum()} of {len(y)} malignant, {y.mean():.1%}\n")

for name, cv in [("KFold          ", KFold(n_splits=10, shuffle=True, random_state=0)),
                 ("StratifiedKFold", StratifiedKFold(n_splits=10, shuffle=True, random_state=0))]:
    counts = [int(y[test_idx].sum()) for _, test_idx in cv.split(X, y)]
    print(name, "malignant in each test fold:", counts)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

iris = load_iris()
print("labels in file order:", iris.target[::10])     # every 10th row

model = LogisticRegression(max_iter=1000)
print("KFold(3), no shuffle:          ", cross_val_score(model, iris.data, iris.target, cv=KFold(n_splits=3)))
print("StratifiedKFold(3), no shuffle:", cross_val_score(model, iris.data, iris.target, cv=StratifiedKFold(n_splits=3)).round(4))

## One CV run is not enough either

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

cv_means = []
for r in range(200):                               # 200 different shuffles
    folds = StratifiedKFold(n_splits=10, shuffle=True, random_state=r)
    cv_means.append(cross_val_score(model, X, y, cv=folds).mean())
cv_means = np.array(cv_means)

print(f"random_state=0, the run above: {cv_means[0]:.4f}")
print(f"lowest of the 200:  {cv_means.min():.4f}  (random_state={cv_means.argmin()})")
print(f"highest of the 200: {cv_means.max():.4f}  (random_state={cv_means.argmax()})")
print(f"\nstandard deviation of the CV mean: {cv_means.std():.4f}")

### Repeated cross-validation

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

folds = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=0)
scores = cross_val_score(model, X, y, cv=folds)

print("number of fold scores:", len(scores))
print("mean of each repeat:  ", scores.reshape(10, 10).mean(axis=1).round(4))
print(f"\nrepeated 10-fold CV (10 x 10): {scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

rep_means = []
for r in range(50):                                # 50 different sets of 10 shuffles
    folds = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=r)
    rep_means.append(cross_val_score(model, X, y, cv=folds).mean())
rep_means = np.array(rep_means)

print(f"repeated CV mean over 50 random_states:  lowest {rep_means.min():.4f}"
      f"   highest {rep_means.max():.4f}")
print(f"standard deviation of the repeated-CV mean: {rep_means.std():.4f}")

## A warning: preprocessing goes inside the fold

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target
folds = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# WRONG: the scaler sees all 569 rows, test folds included, before CV starts
X_scaled = StandardScaler().fit_transform(X)
wrong = cross_val_score(LogisticRegression(max_iter=5000), X_scaled, y, cv=folds)

# RIGHT: the pipeline re-fits the scaler on the 9 training folds, every round
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
right = cross_val_score(model, X, y, cv=folds)

print(f"scaler fitted on all 569 rows: {wrong.mean():.4f}")
print(f"scaler fitted inside each fold: {right.mean():.4f}")

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.linear_model import LogisticRegression

rng = np.random.RandomState(0)
X = rng.normal(size=(100, 5000))        # 100 rows of pure noise, 5000 columns
y = rng.randint(0, 2, size=100)         # labels from a coin flip: nothing to learn
folds = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# WRONG: keep the 20 columns that best match y, chosen using all 100 rows
X_picked = SelectKBest(k=20).fit_transform(X, y)
wrong = cross_val_score(LogisticRegression(), X_picked, y, cv=folds)

# RIGHT: the choice of columns happens inside each fold
right = cross_val_score(make_pipeline(SelectKBest(k=20), LogisticRegression()), X, y, cv=folds)

print(f"columns chosen on all rows:     {wrong.mean():.4f}")
print(f"columns chosen inside each fold: {right.mean():.4f}")

Back to the [Topic 12 lecture note](https://dongchul.kim/csci3320-fall2026/topic/11).